# Testes de Quebras Estruturais

Quebras estruturais ocorrem quando os parametros de um modelo mudam ao longo do tempo. Ignorar quebras pode levar a:
- Estimativas viesadas dos coeficientes
- Previsoes incorretas
- Testes de raiz unitaria com resultado errado (vies de nao-rejeicao)

Este notebook cobre tres abordagens complementares:

1. **Teste de Chow**: testa se os parametros mudam em uma data **conhecida** a priori
2. **CUSUM e CUSUM-sq**: testes de estabilidade **recursiva** que detectam instabilidade sem especificar a data
3. **Bai-Perron**: detecta **multiplas** quebras de forma **endogena** (data desconhecida)

---

## Conteudo

1. Geracao de series com quebra estrutural
2. Teste de Chow (quebra em data conhecida)
3. CUSUM e CUSUM de quadrados
4. Bai-Perron (multiplas quebras endogenas)
5. Aplicacao: PIB do Brasil
6. Visualizacao de pontos de quebra
7. Tabela resumo
8. Exercicios

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronobox.tests_stat import chow_test, cusum_test, cusumsq_test, bai_perron_test

import sys
sys.path.insert(0, '..')
from utils.data_generators import generate_structural_break
from utils.plot_helpers import plot_structural_break, plot_cusum

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Geracao de Series com Quebra Estrutural

Vamos gerar tres tipos de series:
1. **Quebra de nivel**: a media muda abruptamente em t=100
2. **Quebra de tendencia**: a inclinacao muda
3. **Sem quebra**: serie estavel para controle (verificar falsos positivos)

In [ ]:
# Serie com quebra de nivel (shift=3.0 no ponto medio)
df_level = generate_structural_break(n=200, break_point=0.5, shift=3.0, seed=42)
true_break_level = int(df_level['true_break_index'].iloc[0])

# Serie com quebra de tendencia (inclinacao muda)
df_trend = generate_structural_break(n=200, break_point=0.5, shift=0.0, seed=42,
                                      trend_before=0.02, trend_after=0.08)
true_break_trend = int(df_trend['true_break_index'].iloc[0])

# Serie sem quebra (controle)
rng = np.random.default_rng(42)
y_stable = rng.normal(0, 1, 200)

# Visualizacao
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

plot_structural_break(pd.Series(df_level['y'].values), true_break_level,
                      title='Quebra de Nivel', ax=axes[0])
plot_structural_break(pd.Series(df_trend['y'].values), true_break_trend,
                      title='Quebra de Tendencia', ax=axes[1])
axes[2].plot(y_stable, color='steelblue', linewidth=1.0)
axes[2].set_title('Sem Quebra (Controle)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Series para Testes de Quebra Estrutural', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Quebra verdadeira (nivel):    indice {true_break_level}")
print(f"Quebra verdadeira (tendencia): indice {true_break_trend}")

## 2. Teste de Chow

O teste de Chow (1960) verifica se os coeficientes de uma regressao linear mudam em um ponto de quebra **conhecido a priori**.

$$H_0: \text{os coeficientes sao estaveis (sem quebra)}$$
$$H_1: \text{os coeficientes mudam no ponto de quebra}$$

O teste compara a soma de quadrados dos residuos (SQR) do modelo completo com a soma das SQR dos dois submodelos:

$$F = \frac{(SQR_{total} - SQR_1 - SQR_2) / k}{(SQR_1 + SQR_2) / (T - 2k)}$$

**Requerimentos do teste de Chow:**
- `y`: variavel dependente
- `x_mat`: matriz de regressores (incluir constante manualmente)
- `break_point`: indice da quebra (deve ser conhecido!)

In [ ]:
# Preparar matriz de regressores (constante + tendencia)
n = 200
t = np.arange(n)
x_mat = np.column_stack([np.ones(n), t])  # constante + tendencia linear

# Teste de Chow na serie com quebra de nivel - break no ponto correto
print("=== Chow: Quebra de Nivel (ponto correto) ===")
y_level = df_level['y'].values
chow_correct = chow_test(y_level, x_mat, break_point=true_break_level)
print(chow_correct.summary())
print(f"Decisao: {'Quebra detectada!' if chow_correct.reject_at_5pct else 'Sem quebra'}")

print("\n")

# Teste de Chow no ponto ERRADO (t=50, longe da quebra real em t=100)
print("=== Chow: Quebra de Nivel (ponto errado, t=50) ===")
chow_wrong = chow_test(y_level, x_mat, break_point=50)
print(chow_wrong.summary())
print(f"Decisao: {'Quebra detectada' if chow_wrong.reject_at_5pct else 'Sem quebra (correto - ponto errado)'}")

print("\n")

# Controle: Chow na serie estavel (nao deve rejeitar)
print("=== Chow: Serie Estavel (controle) ===")
chow_stable = chow_test(y_stable, x_mat, break_point=100)
print(chow_stable.summary())
print(f"Decisao: {'Falso positivo!' if chow_stable.reject_at_5pct else 'Sem quebra (correto)'}")

## 3. CUSUM e CUSUM de Quadrados

Os testes CUSUM (Brown, Durbin e Evans, 1975) sao baseados em **residuos recursivos** e nao requerem especificacao da data de quebra.

### CUSUM
Acumula os residuos recursivos padronizados:
$$W_t = \sum_{j=k+1}^{t} \frac{w_j}{\hat{\sigma}}$$

Se os parametros sao estaveis, $W_t$ oscila ao redor de zero. Uma quebra faz $W_t$ divergir para fora das bandas criticas.

### CUSUM de Quadrados (CUSUM-sq)
Acumula os **quadrados** dos residuos recursivos:
$$S_t = \frac{\sum_{j=k+1}^{t} w_j^2}{\sum_{j=k+1}^{T} w_j^2}$$

$S_t$ cresce linearmente de 0 a 1 sob estabilidade. Desvios indicam mudanca na variancia dos residuos.

In [ ]:
# CUSUM na serie com quebra de nivel
print("=== CUSUM: Serie com Quebra de Nivel ===")
cusum_level = cusum_test(y_level, x_mat, significance=0.05)
print(cusum_level.summary())

# Extrair dados para visualizacao
if 'cusum_values' in cusum_level.additional_info:
    cusum_vals = cusum_level.additional_info['cusum_values']
    upper = cusum_level.additional_info.get('upper_bound', np.zeros_like(cusum_vals))
    lower = cusum_level.additional_info.get('lower_bound', np.zeros_like(cusum_vals))
    plot_cusum(cusum_vals, upper, lower, title='CUSUM: Serie com Quebra de Nivel')
    plt.show()
else:
    print("(dados do CUSUM nao disponiveis para visualizacao)")

In [ ]:
# CUSUM-sq na serie com quebra de nivel
print("=== CUSUM de Quadrados: Serie com Quebra de Nivel ===")
cusumsq_level = cusumsq_test(y_level, x_mat, significance=0.05)
print(cusumsq_level.summary())

# Visualizacao CUSUM-sq
if 'cusumsq_values' in cusumsq_level.additional_info:
    csq_vals = cusumsq_level.additional_info['cusumsq_values']
    upper_sq = cusumsq_level.additional_info.get('upper_bound', np.zeros_like(csq_vals))
    lower_sq = cusumsq_level.additional_info.get('lower_bound', np.zeros_like(csq_vals))
    plot_cusum(csq_vals, upper_sq, lower_sq, title='CUSUM-sq: Serie com Quebra de Nivel')
    plt.show()

print("\n")

# CUSUM na serie estavel (controle - nao deve rejeitar)
print("=== CUSUM: Serie Estavel (controle) ===")
cusum_stable = cusum_test(y_stable, x_mat, significance=0.05)
print(cusum_stable.summary())
print(f"Decisao: {'Instabilidade detectada' if cusum_stable.reject_at_5pct else 'Estavel (correto)'}")

## 4. Teste de Bai-Perron (Multiplas Quebras Endogenas)

O teste de Bai-Perron (1998, 2003) e o mais poderoso dos tres: detecta **multiplas** quebras estruturais de forma **endogena** (sem conhecer as datas a priori).

O algoritmo usa **programacao dinamica** para encontrar as $m$ quebras que minimizam a soma global de quadrados dos residuos:

$$\min_{t_1, \ldots, t_m} \sum_{j=1}^{m+1} \sum_{t=t_{j-1}+1}^{t_j} (y_t - x_t' \beta_j)^2$$

Parametros:
- `max_breaks`: numero maximo de quebras a testar
- `trim`: tamanho minimo de cada segmento (como fracao de T)

In [ ]:
# Bai-Perron na serie com uma quebra de nivel
print("=== Bai-Perron: Serie com 1 Quebra de Nivel ===")
bp_level = bai_perron_test(y_level, max_breaks=3, trim=0.15)
print(bp_level.summary())

# Extrair quebras detectadas
if 'break_indices' in bp_level.additional_info:
    breaks = bp_level.additional_info['break_indices']
    print(f"\nQuebras detectadas: {breaks}")
    print(f"Quebra verdadeira:  {true_break_level}")

In [ ]:
# Gerar serie com DUAS quebras para testar Bai-Perron
rng = np.random.default_rng(42)
n = 300
y_two_breaks = np.zeros(n)
# Regime 1: t=0..99, media=0
y_two_breaks[:100] = rng.normal(0, 1, 100)
# Regime 2: t=100..199, media=3
y_two_breaks[100:200] = rng.normal(3, 1, 100)
# Regime 3: t=200..299, media=1
y_two_breaks[200:] = rng.normal(1, 1, 100)

print("=== Bai-Perron: Serie com 2 Quebras ===")
x_mat_300 = np.ones((n, 1))  # apenas constante
bp_two = bai_perron_test(y_two_breaks, x_mat=x_mat_300, max_breaks=5, trim=0.15)
print(bp_two.summary())

if 'break_indices' in bp_two.additional_info:
    breaks = bp_two.additional_info['break_indices']
    print(f"\nQuebras detectadas: {breaks}")
    print(f"Quebras verdadeiras: [100, 200]")

In [ ]:
# Visualizacao das quebras detectadas pelo Bai-Perron
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Serie com 1 quebra
axes[0].plot(y_level, color='steelblue', linewidth=1.0)
axes[0].axvline(x=true_break_level, color='red', linestyle='--', linewidth=2, label=f'Verdadeira ({true_break_level})')
if 'break_indices' in bp_level.additional_info:
    for bi in bp_level.additional_info['break_indices']:
        axes[0].axvline(x=bi, color='green', linestyle=':', linewidth=2, label=f'Estimada ({bi})')
axes[0].set_title('1 Quebra: Verdadeira vs Estimada')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Serie com 2 quebras
axes[1].plot(y_two_breaks, color='steelblue', linewidth=1.0)
for true_b in [100, 200]:
    axes[1].axvline(x=true_b, color='red', linestyle='--', linewidth=2,
                    label=f'Verdadeira ({true_b})' if true_b == 100 else f'Verdadeira ({true_b})')
if 'break_indices' in bp_two.additional_info:
    for bi in bp_two.additional_info['break_indices']:
        axes[1].axvline(x=bi, color='green', linestyle=':', linewidth=2, label=f'Estimada ({bi})')
axes[1].set_title('2 Quebras: Verdadeiras vs Estimadas')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Bai-Perron: Deteccao de Quebras Estruturais', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Aplicacao: PIB do Brasil (brazil_gdp.csv)

O PIB do Brasil possui uma quebra estrutural conhecida por volta de 2003 (inicio do governo Lula, boom de commodities). Vamos aplicar todos os testes para detecta-la.

In [ ]:
# Carregar PIB do Brasil
gdp_br = pd.read_csv('../data/brazil_gdp.csv', parse_dates=['date'], index_col='date')

# Usar taxa de crescimento (estacionaria) para os testes de quebra
growth_br = gdp_br['gdp_growth'].dropna()
y_br = growth_br.values
n_br = len(y_br)
t_br = np.arange(n_br)
x_mat_br = np.column_stack([np.ones(n_br), t_br])

# Visualizar
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(growth_br.index, growth_br.values, color='coral', linewidth=1.0)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax.set_title('Taxa de Crescimento Trimestral do PIB Brasil (%)')
ax.set_ylabel('Crescimento (%)')
ax.grid(True, alpha=0.3)

# Marcar 2003 Q1
if pd.Timestamp('2003-01-01') in growth_br.index:
    ax.axvline(x=pd.Timestamp('2003-01-01'), color='red', linestyle='--',
               linewidth=2, label='2003-Q1 (quebra esperada)')
    ax.legend()
plt.tight_layout()
plt.show()

print(f"Observacoes: {n_br}")
print(f"Periodo: {growth_br.index[0].strftime('%Y-%m')} a {growth_br.index[-1].strftime('%Y-%m')}")

In [ ]:
# Bateria completa de testes no PIB Brasil
print("=" * 70)
print("  Testes de Quebra Estrutural: PIB Brasil (crescimento)")
print("=" * 70)

# Chow no indice correspondente a ~2003 Q1 (indice 36 = trimestre 36 desde 1994 Q2)
# A quebra nos dados sinteticos esta em break_q=36
chow_br = chow_test(y_br, x_mat_br, break_point=36)
print("\n--- Chow (break=36, ~2003-Q1) ---")
print(f"  F-stat: {chow_br.statistic:.4f}, p={chow_br.pvalue:.4f}")
print(f"  Quebra: {'Sim' if chow_br.reject_at_5pct else 'Nao'}")

# CUSUM
cusum_br = cusum_test(y_br, x_mat_br, significance=0.05)
print("\n--- CUSUM ---")
print(f"  Estatistica: {cusum_br.statistic:.4f}")
print(f"  Instabilidade: {'Sim' if cusum_br.reject_at_5pct else 'Nao'}")

# CUSUM-sq
cusumsq_br = cusumsq_test(y_br, x_mat_br, significance=0.05)
print("\n--- CUSUM-sq ---")
print(f"  Estatistica: {cusumsq_br.statistic:.4f}")
print(f"  Instabilidade: {'Sim' if cusumsq_br.reject_at_5pct else 'Nao'}")

# Bai-Perron
bp_br = bai_perron_test(y_br, max_breaks=3, trim=0.15)
print("\n--- Bai-Perron ---")
print(bp_br.summary())
if 'break_indices' in bp_br.additional_info:
    breaks_br = bp_br.additional_info['break_indices']
    print(f"  Quebras detectadas (indices): {breaks_br}")
    for bi in breaks_br:
        if bi < len(growth_br):
            print(f"    indice {bi} -> {growth_br.index[bi].strftime('%Y-%m')}")

## 6. Visualizacao dos Pontos de Quebra Estimados no PIB Brasil

In [ ]:
# Visualizacao consolidada das quebras no PIB Brasil
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Painel 1: Log PIB com quebras Bai-Perron
axes[0].plot(gdp_br.index, gdp_br['log_gdp'], color='coral', linewidth=1.2)
axes[0].set_title('Log PIB Brasil: Quebras Estruturais Estimadas (Bai-Perron)')
axes[0].set_ylabel('Log(PIB)')
axes[0].grid(True, alpha=0.3)

if 'break_indices' in bp_br.additional_info:
    for i, bi in enumerate(bp_br.additional_info['break_indices']):
        if bi < len(growth_br):
            date = growth_br.index[bi]
            # Mapear para o indice do log_gdp (offset de 1 por causa do dropna)
            axes[0].axvline(x=date, color='green', linestyle='--', linewidth=2,
                          label=f'Quebra {i+1}: {date.strftime("%Y-%m")}')
            axes[1].axvline(x=date, color='green', linestyle='--', linewidth=2,
                          label=f'Quebra {i+1}: {date.strftime("%Y-%m")}')

# Painel 2: Taxa de crescimento com quebras
axes[1].plot(growth_br.index, growth_br.values, color='coral', linewidth=1.0, alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_title('Crescimento PIB Brasil (%): Quebras Bai-Perron')
axes[1].set_ylabel('Crescimento (%)')
axes[1].grid(True, alpha=0.3)

# Colorir regimes distintos se existirem quebras
if 'break_indices' in bp_br.additional_info:
    breaks_idx = bp_br.additional_info['break_indices']
    colors = ['steelblue', 'coral', 'green', 'purple']
    all_breaks = [0] + list(breaks_idx) + [n_br]
    for j in range(len(all_breaks) - 1):
        start, end = all_breaks[j], all_breaks[j + 1]
        regime_data = growth_br.iloc[start:end]
        mean_val = regime_data.mean()
        axes[1].axhline(y=mean_val, xmin=start / n_br, xmax=end / n_br,
                        color=colors[j % len(colors)], linestyle=':', linewidth=2, alpha=0.7)

for ax in axes:
    ax.legend(loc='best')

plt.tight_layout()
plt.show()

## 7. Tabela Resumo de Resultados

In [ ]:
# Tabela resumo consolidada
summary = []

# Series sinteticas
for name, chow_r, cusum_r, bp_r, true_bp in [
    ("Quebra nivel (sint.)", chow_correct, cusum_level, bp_level, str(true_break_level)),
    ("Estavel (controle)", chow_stable, cusum_stable, None, "N/A"),
]:
    row = {
        'Serie': name,
        'Chow stat': f"{chow_r.statistic:.3f}",
        'Chow p': f"{chow_r.pvalue:.3f}",
        'Chow rejeita': chow_r.reject_at_5pct,
        'CUSUM rejeita': cusum_r.reject_at_5pct,
        'Quebra verdadeira': true_bp,
    }
    if bp_r and 'break_indices' in bp_r.additional_info:
        row['BP quebras'] = str(bp_r.additional_info['break_indices'])
    else:
        row['BP quebras'] = 'N/A' if bp_r is None else str(bp_r.additional_info.get('break_indices', []))
    summary.append(row)

# PIB Brasil
summary.append({
    'Serie': 'PIB Brasil (cresc.)',
    'Chow stat': f"{chow_br.statistic:.3f}",
    'Chow p': f"{chow_br.pvalue:.3f}",
    'Chow rejeita': chow_br.reject_at_5pct,
    'CUSUM rejeita': cusum_br.reject_at_5pct,
    'BP quebras': str(bp_br.additional_info.get('break_indices', [])),
    'Quebra verdadeira': '~36 (2003-Q1)',
})

df_summary = pd.DataFrame(summary)
print("=" * 95)
print("  TABELA RESUMO: Testes de Quebra Estrutural")
print("=" * 95)
print(df_summary.to_string(index=False))

## Exercicio

### Exercicio 1: Sensibilidade do Bai-Perron ao Trim

Usando a serie com 2 quebras (`y_two_breaks`):
1. Aplique Bai-Perron com `trim` = 0.05, 0.10, 0.15, 0.20, 0.25
2. Como o tamanho minimo do segmento afeta o numero de quebras detectadas?
3. Qual valor de `trim` voce recomendaria?

**Dica**: `bai_perron_test(y, max_breaks=5, trim=t)`. Valores menores de trim permitem segmentos menores.

In [ ]:
# Exercicio 1 - Seu codigo aqui
# Teste Bai-Perron com diferentes valores de trim


### Exercicio 2: Detectando Quebras no PIB dos EUA

Carregue `us_gdp_quarterly.csv` e:
1. Aplique Chow no ponto correspondente a 2008 (crise financeira) e a 2020 (COVID)
2. Aplique Bai-Perron com `max_breaks=3` na taxa de crescimento
3. O Bai-Perron detecta as crises de 2008 e 2020?
4. Visualize as quebras estimadas

**Dica**: Use a taxa de crescimento (`gdp_growth`) para os testes

In [ ]:
# Exercicio 2 - Seu codigo aqui
# Carregue us_gdp_quarterly.csv e aplique os testes de quebra
